In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.chdir('..')

In [4]:
from sklearn.metrics import f1_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from pathlib import Path
from argparse import ArgumentParser

import ray
import time
import torch
import json
import numpy as np

from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset

from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN
from cluster_intrep_repo.stacks_utils import *
from tqdm.auto import tqdm, trange

In [5]:
def timing_decorator(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"{func.__name__} took {execution_time:.4f} seconds to execute")
        return result
    return wrapper

In [6]:
ray.init(address="auto", namespace="blocksworld")

2025-03-12 23:29:54,552	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.61.4.10:6379...
2025-03-12 23:29:54,562	INFO worker.py:1832 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


Python version:,3.11.11
Ray version:,2.42.1
Dashboard:,http://127.0.0.1:8265


(RayTrainWorker pid=3107361) Setting up process group for: env:// [rank=0, world_size=8]
(TorchTrainer pid=3107170) Started distributed worker processes: 
(TorchTrainer pid=3107170) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3107361) world_rank=0, local_rank=0, node_rank=0
(TorchTrainer pid=3107170) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3107362) world_rank=1, local_rank=1, node_rank=0
(TorchTrainer pid=3107170) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3107360) world_rank=2, local_rank=2, node_rank=0
(TorchTrainer pid=3107170) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3107357) world_rank=3, local_rank=3, node_rank=0
(TorchTrainer pid=3107170) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3107359) world_rank=4, local_rank=4, node_rank=0
(TorchTrainer pid=3107170

(RayTrainWorker pid=3112006) Batch time: 0.5379 seconds
(RayTrainWorker pid=3112007) Epoch 0, Train Loss: 566.6735, Hits: 0.2000, F1: 0.1667, Val Loss: 447.8152
(RayTrainWorker pid=3112007) Epoch 1, Train Loss: 538.1808, Hits: 0.2000, F1: 0.1333, Val Loss: 430.6847
(RayTrainWorker pid=3112007) Epoch 2, Train Loss: 514.8031, Hits: 0.2000, F1: 0.1333, Val Loss: 411.9619
(RayTrainWorker pid=3112007) Epoch 3, Train Loss: 493.4028, Hits: 0.2000, F1: 0.1333, Val Loss: 394.9400
(RayTrainWorker pid=3112007) Epoch 4, Train Loss: 474.8393, Hits: 0.4000, F1: 0.3556, Val Loss: 381.6584
(RayTrainWorker pid=3112007) Epoch 5, Train Loss: 457.7919, Hits: 0.6000, F1: 0.4444, Val Loss: 369.1046
(RayTrainWorker pid=3112007) Epoch 6, Train Loss: 443.1941, Hits: 0.8000, F1: 0.6190, Val Loss: 355.9553
(RayTrainWorker pid=3112007) Epoch 7, Train Loss: 429.5313, Hits: 0.4000, F1: 0.2857, Val Loss: 345.3359
(RayTrainWorker pid=3112007) Epoch 8, Train Loss: 417.7910, Hits: 0.0000, F1: 0.0000, Val Loss: 336.1778

(RayTrainWorker pid=3133284) Setting up process group for: env:// [rank=0, world_size=8]
(TorchTrainer pid=3133164) Started distributed worker processes: 
(TorchTrainer pid=3133164) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3133284) world_rank=0, local_rank=0, node_rank=0
(TorchTrainer pid=3133164) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3133282) world_rank=1, local_rank=1, node_rank=0
(TorchTrainer pid=3133164) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3133286) world_rank=2, local_rank=2, node_rank=0
(TorchTrainer pid=3133164) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3133283) world_rank=3, local_rank=3, node_rank=0
(TorchTrainer pid=3133164) - (node_id=35a1a240cc60f3fe93775a565d80243413fb585365ef4310fbe3d201, ip=10.61.4.10, pid=3133285) world_rank=4, local_rank=4, node_rank=0
(TorchTrainer pid=3133164

(RayTrainWorker pid=3133284) Batch time: 1.7158 seconds
(RayTrainWorker pid=3133284) Epoch 0, Train Loss: 518.2953, Hits: 0.4000, F1: 0.3000, Val Loss: 433.1779
(RayTrainWorker pid=3133284) Epoch 1, Train Loss: 492.7968, Hits: 0.4000, F1: 0.3000, Val Loss: 416.6317
(RayTrainWorker pid=3133284) Epoch 2, Train Loss: 471.4338, Hits: 0.4000, F1: 0.3556, Val Loss: 401.0501
(RayTrainWorker pid=3133284) Epoch 3, Train Loss: 452.7147, Hits: 0.6000, F1: 0.5833, Val Loss: 386.8536
(RayTrainWorker pid=3133284) Epoch 4, Train Loss: 435.5374, Hits: 0.8000, F1: 0.7619, Val Loss: 373.2960
(RayTrainWorker pid=3133284) Epoch 5, Train Loss: 420.0634, Hits: 0.6000, F1: 0.5833, Val Loss: 361.5466
(RayTrainWorker pid=3133284) Epoch 6, Train Loss: 407.0320, Hits: 0.2000, F1: 0.1333, Val Loss: 351.2677
(RayTrainWorker pid=3133284) Epoch 7, Train Loss: 396.0472, Hits: 0.4000, F1: 0.4000, Val Loss: 340.9710
(RayTrainWorker pid=3133284) Epoch 8, Train Loss: 385.5990, Hits: 0.6000, F1: 0.5833, Val Loss: 331.6135

(raylet) [2025-03-12 23:56:24,026 E 4023721 4023753] file_system_monitor.cc:116: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162 is over 95% full, available space: 4.69836 GB; capacity: 387.482 GB. Object creation will fail if spilling is required.
(raylet) [2025-03-12 23:56:25,280 E 4023721 4023752] dlmalloc.cc:202: Out of disk space with fallocate error: No space left on device
(raylet) [2025-03-12 23:56:25,280 E 4023721 4023752] object_lifecycle_manager.cc:214: Plasma fallback allocator failed, likely out of disk space.
(bundle_reservation_check_func pid=4024541) [*** LOG ERROR #0001 ***] [2025-03-12 23:56:31] [ray_log_sink] Failed writing to file /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/logs/python-core-worker-214b746f2a8f65e0c2525e430e18b3608aa6234d082d627664f4c41c_4024541.log: No space left on device
(raylet) [2025-03-12 23:56:34,032 E 4023721 4023753] file_system_monitor.cc:116: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162 is over 95% full, available space:

In [7]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device = 'cuda'
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = initialize_tokenizer(model_id)


In [8]:
n_blocks = 6

In [9]:
parsed_datasets = {
    4: "blocksworld-4-blocks-actions-first.json",
    6: "blocksworld-6-blocks-actions-first.json"
}

In [10]:
cur_dir = Path(".").absolute()

In [11]:
def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/deepseek-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)
        

def load_datasets():
    dataset = load_dataset(
    f"dmitriihook/deepseek-r1-qwen-32b-planning-{blocksworld_type[n_blocks]}")["train"]
    
    task_name = "plan_generation_po"
    domain_name = f"blocksworld_{n_blocks}_blocks"
    eval_results = load_dataset_from_file(domain_name, task_name)["instances"]
    eval_results = {x["dataset_idx"]: x for x in eval_results}

    with open(cur_dir / "first_action_token_logits.json") as f:
        labels_dataset = json.load(f)

    labels_dataset = {
        int(k): v for k, v in labels_dataset.items()
    }

    return dataset, eval_results, labels_dataset

In [12]:
def make_labels_dict(labels_dataset, dataset):
    all_blocks = [
        chr(ord('A') + i) for i in range(n_blocks)
    ]

    n_rows = row_ns[n_blocks]
    n_rows = 5000

    labels_dict = defaultdict(dict)

    for idx, row in enumerate(dataset.select(range(n_rows))):
        generation = row["generation"]

        steps = generation.split("\n\n")[:50]

        if idx not in labels_dataset:
            continue

        for line_n, step in enumerate(steps):
            logits = []
            for block in all_blocks:
                logits.append(labels_dataset[idx]["block_logits"][block][line_n])

            logits = np.array(logits)
            amax = np.argmax(logits)

            labels_dict[idx][line_n] = {
                "logits": logits,
                "max_block": all_blocks[amax],
                "step": step
            }

    return labels_dict

In [13]:
# total_layers = model.config.num_hidden_layers
total_layers = 64

In [14]:
def make_data_to_process(dataset, labels_dict, n_rows, eval_results, answer_type, tokenizer):
    data_to_process = []
    for idx, row in enumerate(dataset.select(range(n_rows))):
        if eval_results[idx]["llm_correct"] and answer_type == "incorrect":
            continue
        if not eval_results[idx]["llm_correct"] and answer_type == "correct":
            continue
        generation = row["generation"]

        if idx not in labels_dict:
            continue

        # need to remove the eos token
        pos_start = len(tokenize_blocksworld_generation(tokenizer, row, "")[0][:-1])

        pos_pre = pos_start

        for line_n, line in enumerate(generation.split("\n\n")[:50]):
            line_tokens = tokenizer.tokenize("\n\n" + line + "\n\n")[1:]
            pos_post = pos_pre + len(line_tokens)

            data_to_process.append({
                "idx": idx,
                "line_n": line_n,
                "pos_pre": pos_pre,
                "pos_post": pos_post,
                "pos_start": pos_start,
                "logits": labels_dict[idx][line_n]["logits"],
                "label": labels_dict[idx][line_n]["max_block"],
            })
            pos_pre = pos_post


    return data_to_process

In [15]:
def process_data(items, n_blocks, max_tokens = 3200):
    new_items = []

    for item in items:

        block = item["label"]

        if item["pos_post"] >= max_tokens:
            continue

        try:
            label = block2int(block, n_blocks)
        except Exception as e:
            print(e)
            continue

        new_items.append({
            "pos_pre": item["pos_pre"],
            "pos_post": item["pos_post"],
            "pos_start": item["pos_start"],
            "label": label,
            "idx":  item["idx"],
            "line_n": item["line_n"],
            "logits": item["logits"],
            "label": label
        })

    return new_items
    

In [16]:
def collate_fn(batch):
    inputs = [torch.tensor(x) for x in batch["input"]]    
    masks = [torch.ones(x.shape[0], dtype=torch.bool) for x in inputs]
    inputs = pad_sequence(inputs, batch_first=True,
                          padding_value=0, padding_side="left")
    masks = pad_sequence(masks, batch_first=True,
                         padding_value=True, padding_side="left")
    labels = np.stack([x for x in batch["labels"]])
    labels = torch.tensor(labels, dtype=torch.int64)

    logits = np.stack([x for x in batch["logits"]])
    logits = torch.tensor(logits, dtype=torch.float32)
    return {
        "input": inputs.to(device),
        "labels": labels.to(device),
        "logits": logits.to(device),
        "mask": masks.to(device)
    }

In [17]:
class StepProbeDataset(Dataset):
    def __init__(self, items, n_layer, n_prev_tokens, shift_tokens, dataset_actor_name, n_blocks, batch_size):
        self.items = process_data(items, n_blocks)
        self.n_layer = n_layer
        self.n_blocks = n_blocks
        self.n_prev_tokens = n_prev_tokens
        self.shift_tokens = shift_tokens
        self.dataset_actor = ray.get_actor(dataset_actor_name)
        self.batch_size = batch_size

    def get_batch(self, idxs):
        items = [self.items[idx] for idx in idxs]

        _hidden_states = ray.get(self.dataset_actor.get_batch_layer.remote([item["idx"] for item in items], self.n_layer))

        inputs = []
        labels = []
        logits = []

        for item in items:
            pos_pre = item["pos_pre"]
            pos_post = item["pos_post"]
            pos_start = item["pos_start"]

            pos = pos_post

            label = item["label"]
            _logits = item["logits"]
            hidden_states = _hidden_states[item["idx"]]

            inputs.append(hidden_states[pos - self.shift_tokens - self.n_prev_tokens:pos - self.shift_tokens + 1])
            labels.append(label)
            logits.append(_logits)

        return {
            "input": inputs,
            "labels": labels,
            "logits": logits
        }
        

    def __len__(self):
        return len(self.items) // self.batch_size
    
    def __getitem__(self, idx):
        start_item_idx = idx * self.batch_size
        end_item_idx = min((idx + 1) * self.batch_size, len(self.items))

        batch = self.get_batch(range(start_item_idx, end_item_idx))

        batch = collate_fn(batch)

        return batch

In [43]:
class DataFetcher:
    def __init__(self, dataset: StepProbeDataset, macro_batch_size: int, rank: int, shuffle: bool = True):
        self.dataset = dataset
        self.macro_batch_size = macro_batch_size
        self.total_items = len(dataset.items)
        self.item_ids = np.arange(self.total_items)
        self.rank = rank
        if shuffle:
            self.item_ids = np.random.permutation(self.item_ids)
        self.item_ids = np.array_split(self.item_ids, self.total_items  // macro_batch_size)[rank]
        print(f"Rank {rank} has {len(self.item_ids)} items")
        
    def iter_batches(self):
        for macro_batch_start in range(0, len(self.item_ids), self.macro_batch_size):
            macro_batch = self.item_ids[macro_batch_start:macro_batch_start + self.macro_batch_size]
            data = self.dataset.get_batch(macro_batch)
            data_keys = data.keys()

            for mini_batch_start in range(0, len(macro_batch), self.dataset.batch_size):
                mini_batch = {
                    k: data[k][mini_batch_start:mini_batch_start + self.dataset.batch_size] for k in data_keys
                }                
                mini_batch = collate_fn(mini_batch)

                yield mini_batch


In [20]:
class StepProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        # self.fc = torch.nn.Linear(input_size, hidden_size)
        # self.fc2 = torch.nn.Linear(hidden_size, n_blocks * (n_blocks + 2) * 2)
        # self.fc2 = torch.nn.Linear(input_size, n_blocks * (n_blocks + 2) * 2)
        self.fc2 = torch.nn.Linear(input_size, n_blocks)
        # self.dropout = torch.nn.Dropout(0.1)

    def forward(self, x):
        # x = self.fc(x)
        # x = torch.nn.functional.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x
        # return x.view(-1, n_blocks + 2, n_blocks * 2)


class GRUProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        self.gru = torch.nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x, *args):
        x, _ = self.gru(x)
        x = self.fc(x[:, -1])
        return x
    
class MultiProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_probes):
        super().__init__()
        self.probes = torch.nn.ModuleList([torch.nn.Linear(input_size, hidden_size) for _ in range(n_probes)])
        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x):
        
        for i in range(len(self.probes)):
            z_ = self.probes[i](x[:, i])
            if i == 0:
                z = z_
            else:
                z = z + z_
        z = z / len(self.probes)

        z = self.fc(z)

        return z
    
class AHProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        self.q = torch.nn.Parameter(torch.randn(hidden_size))
        self.v = torch.nn.Parameter(torch.randn(hidden_size))

        self.proj = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, 2)

    def forward(self, x, mask):
        x = self.proj(x)
        # scores = torch.einsum("bsh,h->bs", x, self.q)

        scores = x @ self.q

        # print(scores.shape, mask.shape)
        scores = scores.masked_fill(mask, -1000)
        scores = torch.nn.functional.softmax(scores, dim=-1)
        z = torch.matmul(scores.unsqueeze(1), x).squeeze(1)
        z = self.fc(z)
        return z
    
class MLHAProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_heads, n_blocks):
        super().__init__()
        self.head_dim = hidden_size // n_heads
        self.n_heads = n_heads

        self.q = torch.nn.Parameter(torch.randn(n_heads, self.head_dim))

        self.proj_k = torch.nn.Linear(input_size, hidden_size)
        self.proj_v = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x, mask):
        v = self.proj_v(x)
        x = self.proj_k(x)
        x = x.view(x.shape[0], x.shape[1], self.n_heads, self.head_dim)
        v = v.view(v.shape[0], v.shape[1], self.n_heads, self.head_dim)
        scores = torch.einsum("bshd,hd->bsh", x, self.q)

        scores = scores / np.sqrt(self.head_dim)

        scores = scores.masked_fill(mask.unsqueeze(-1), -1000)
        scores = torch.nn.functional.softmax(scores, dim=-2)

        z = torch.einsum("bshd,bsh->bhd", v, scores)
        z = z.view(z.shape[0], -1)
        z = self.fc(z)

        return z


In [ ]:
import ray.train.torch

def train_func(config):
    n_rows = config.get("n_rows", 5000)
    train_test_split = config.get("train_test_split", 0.8)
    n_dim = config.get("n_dim", 5120)
    n_blocks = config.get("n_blocks", 6)
    lr = config.get("lr", 1e-4)
    patience = config.get("patience", 10)
    n_epochs = config.get("n_epochs", 500)
    n_prev_tokens = config.get("n_prev_tokens", 100)
    n_shift_tokens = config.get("n_shift_tokens", 0)
    dataset_actor_name = config.get("dataset_actor_name", "dataset_actor")
    batch_size = config.get("batch_size", 2048)
    macro_batch_size = config.get("macro_batch_size", 10000)
    n_layer = config.get("n_layer", 63)
    
    rank = ray.train.get_context().get_world_rank()
    world_size = ray.train.get_context().get_world_size()

    tokenizer = initialize_tokenizer(model_id)
    dataset, eval_results, labels_dataset = load_datasets()

    labels_dict = make_labels_dict(labels_dataset, dataset)

    training_data = make_data_to_process(dataset, labels_dict, n_rows, eval_results, "all", tokenizer)

    n_train = int(len(training_data) * train_test_split)

    train_items = training_data[:n_train]
    test_items = training_data[n_train:]

    probe = MLHAProbe(n_dim, n_dim, 40, n_blocks)
    probe = GRUProbe(n_dim, 1000, n_blocks)
    probe = ray.train.torch.prepare_model(probe)

    train_dataset = StepProbeDataset(
    train_items, n_layer, n_prev_tokens, n_shift_tokens, dataset_actor_name, n_blocks, batch_size)

    test_dataset = StepProbeDataset(test_items, n_layer, n_prev_tokens, n_shift_tokens, dataset_actor_name, n_blocks, batch_size)

    optimizer = Adam(probe.parameters(), lr=lr)
    criterion = CrossEntropyLoss()
    criterion = torch.nn.MSELoss()


    train_loader = DataFetcher(train_dataset, macro_batch_size, rank)
    test_loader = DataFetcher(test_dataset, macro_batch_size, rank, shuffle=False)


    best_f1 = float('-inf')
    early_stop_counter = 0

    for epoch in range(n_epochs):
        probe.train()
        total_loss = 0
        n_samples = 0

        prev_time = time.time()
        for batch in train_loader.iter_batches():

            optimizer.zero_grad()
            input = batch["input"].float().to(probe.device)
            labels = batch["labels"].to(probe.device)
            labels = batch["logits"].to(probe.device)
            mask = batch["mask"].to(probe.device)

            output = probe(input, mask)

            # print(output.shape, labels.shape, input.shape)

            loss = criterion(output, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch["input"])
            n_samples += len(batch["input"])

            next_time = time.time()

            print(
                f"Batch time: {next_time - prev_time:.4f} seconds"
            )

        avg_train_loss = total_loss / n_samples

        # Evaluation
        probe.eval()
        with torch.no_grad():
            # block_wise_hits = np.zeros((n_blocks * 2), dtype=np.int64)
            block_wise_hits = 0
            total = 0
            val_loss = 0
            all_preds = []
            all_labels = []

            prev_time = time.time()
            for batch in test_loader.iter_batches():
                # print(batch["input"].shape)
                input = batch["input"].float().to(probe.device)
                labels = batch["labels"].to(probe.device)
                labels = batch["logits"].to(probe.device)
                mask = batch["mask"].to(probe.device)

                output = probe(input, mask)

                loss = criterion(output, labels)
                val_loss += loss.item() * len(batch["input"])

                labels = batch["labels"]

                preds = output.argmax(dim=1)  # Assuming classification task
                hits = (preds == labels)

                block_wise_hits += hits.sum(dim=0).cpu().numpy()
                total += len(labels)

                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                

            block_wise_hits = block_wise_hits / total

            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)

            # Compute F1 score block-wise
            # block_wise_f1 = np.zeros(n_blocks * 2)
            # for i in range(n_blocks * 2):
            #     block_wise_f1[i] = f1_score(all_labels[:, i], all_preds[:, i], average='macro')

            # avg_f1 = block_wise_f1.mean()
            avg_f1 = f1_score(all_labels, all_preds, average='macro')

            val_loss /= total
            
            if ray.train.get_context().get_world_rank() == 0:
                print(
                    f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Hits: {block_wise_hits.mean():.4f}, F1: {avg_f1:.4f}, Val Loss: {val_loss:.4f}")

            # Early Stopping Check
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    


In [22]:
n_rows = 5000
train_test_split = 0.8
n_dim = 5120
n_blocks = 6
lr = 1e-4
patience = 10
n_epochs = 500
n_prev_tokens = 100
n_shift_tokens = 0
dataset_actor_name = "dataset_actor"

n_layer = 63

tokenizer = initialize_tokenizer(model_id)
dataset, eval_results, labels_dataset = load_datasets()

labels_dict = make_labels_dict(labels_dataset, dataset)

training_data = make_data_to_process(dataset, labels_dict, n_rows, eval_results, "all", tokenizer)

In [24]:
n_train = int(len(training_data) * train_test_split)

train_items = training_data[:n_train]
test_items = training_data[n_train:]

probe = MLHAProbe(n_dim, n_dim, 40, n_blocks)
probe = GRUProbe(n_dim, 1000, n_blocks).to(device)
# probe = ray.train.torch.prepare_model(probe)

train_dataset = StepProbeDataset(
    train_items, n_layer, n_prev_tokens, n_shift_tokens, dataset_actor_name, n_blocks, 1024)
test_dataset = StepProbeDataset(test_items, n_layer, n_prev_tokens, n_shift_tokens, dataset_actor_name, n_blocks, 1024)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from itertools import islice

def test_fetcher():
    fetcher = DataFetcher(train_dataset, 10000, 0)
    times = []
    prev_time = time.time()
    for batch in islice(fetcher.iter_batches(), 30):
        next_time = time.time()
        times.append(next_time - prev_time)
        
        print(f"Avg time: {np.mean(times):.4f} seconds")

        prev_time = time.time()
        

test_fetcher()

Rank 0 has 10290 items


RayTaskError(OutOfDiskError): [36mray::HiddenStatesDataset.get_batch_layer()[39m (pid=4024543, ip=10.61.4.10, actor_id=459206fbf8421f27d960ccd80c000000, repr=<create_dataset.HiddenStatesDataset object at 0x7f8bebafe3d0>)
  File "python/ray/includes/common.pxi", line 79, in ray._raylet.check_status
ray.exceptions.OutOfDiskError: Local disk is full
The object cannot be created because the local object store is full and the local disk's utilization is over capacity (95% by default).Tip: Use `df` on this node to check disk usage and `ray memory` to check object store memory usage.

[*** LOG ERROR #0001 ***] [2025-03-12 23:56:54] [ray_log_sink] Failed writing to file /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/logs/python-core-driver-16000000ffffffffffffffffffffffffffffffffffffffffffffffff_3079121.log: No space left on device
[*** LOG ERROR #0002 ***] [2025-03-12 23:57:54] [ray_log_sink] Failed writing to file /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/logs/python-core-driver-16000000ffffffffffffffffffffffffffffffffffffffffffffffff_3079121.log: No space left on device
[*** LOG ERROR #0003 ***] [2025-03-12 23:58:54] [ray_log_sink] Failed writing to file /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/logs/python-core-driver-16000000ffffffffffffffffffffffffffffffffffffffffffffffff_3079121.log: No space left on device
[*** LOG ERROR #0004 ***] [2025-03-12 23:59:52] [ray_log_sink] Failed flush to file /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/logs/python-core-driver-16000000ffffffffffffffffffffffffffffffffffffffffffffffff_3079121.log: No

In [32]:
config = {
    "batch_size": 512,
}

scaling_config = ray.train.ScalingConfig(num_workers=8, use_gpu=True)

# [5] Launch distributed training job.
trainer = ray.train.torch.TorchTrainer(
    train_func,
    scaling_config=scaling_config,
    train_loop_config=config,
)
result = trainer.fit()

2025-03-12 23:48:41,602	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2025-03-12 23:48:41,605	WARNING callback.py:136 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`


== Status ==
Current time: 2025-03-12 23:48:41 (running for 00:00:00.11)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/128 CPUs, 8.0/8 GPUs (0.0/1.0 accelerator_type:H100)
Result logdir: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/artifacts/2025-03-12_23-48-41/TorchTrainer_2025-03-12_23-48-41/driver_artifacts
Number of trials: 1/1 (1 PENDING)


== Status ==
Current time: 2025-03-12 23:48:46 (running for 00:00:05.15)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/128 CPUs, 8.0/8 GPUs (0.0/1.0 accelerator_type:H100)
Result logdir: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/artifacts/2025-03-12_23-48-41/TorchTrainer_2025-03-12_23-48-41/driver_artifacts
Number of trials: 1/1 (1 RUNNING)


== Status ==
Current time: 2025-03-12 23:48:51 (running for 00:00:10.17)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/128 CPUs, 8.0/8 GPUs (0.0/1.0 accelerator_type:H100)
Result logdir: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/artifa

2025-03-12 23:51:43,326	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-03-12 23:51:43,330	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/nebius/ray_results/TorchTrainer_2025-03-12_23-48-41' in 0.0024s.


== Status ==
Current time: 2025-03-12 23:51:43 (running for 00:03:01.72)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/128 CPUs, 8.0/8 GPUs (0.0/1.0 accelerator_type:H100)
Result logdir: /tmp/ray/session_2025-03-12_15-44-02_243850_4023162/artifacts/2025-03-12_23-48-41/TorchTrainer_2025-03-12_23-48-41/driver_artifacts
Number of trials: 1/1 (1 RUNNING)




2025-03-12 23:51:53,335	INFO tune.py:1041 -- Total run time: 191.73 seconds (181.72 seconds for the tuning loop).
2025-03-12 23:51:53,336	WARNING tune.py:1051 -- Training has been interrupted, but the most recent state was saved.
Resume training with: <FrameworkTrainer>.restore(path="/home/nebius/ray_results/TorchTrainer_2025-03-12_23-48-41", ...)


In [ ]:
training_data = data_all


def train_probe(probe, train_dataset, test_dataset, patience=30, lr=1e-4):
    optimizer = Adam(probe.parameters(), lr=lr)
    criterion = CrossEntropyLoss()
    criterion = torch.nn.MSELoss()
    train_loader = DataLoader(
        train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn, drop_last=False)
    test_loader = DataLoader(
        test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn, drop_last=False)

    n_epochs = 500
    best_f1 = float('-inf')
    early_stop_counter = 0

    for epoch in range(n_epochs):
        probe.train()
        total_loss = 0
        n_samples = 0

        for batch in train_loader:

            optimizer.zero_grad()
            input = batch["input"].to(device).float()
            labels = batch["labels"].to(device)
            labels = batch["logits"].to(device)
            mask = batch["mask"].to(device)

            output = probe(input, mask)

            # print(output.shape, labels.shape, input.shape)

            loss = criterion(output, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch["input"])
            n_samples += len(batch["input"])

        avg_train_loss = total_loss / n_samples

        # Evaluation
        probe.eval()
        with torch.no_grad():
            # block_wise_hits = np.zeros((n_blocks * 2), dtype=np.int64)
            block_wise_hits = 0
            total = 0
            val_loss = 0
            all_preds = []
            all_labels = []

            for batch in test_loader:
                # print(batch["input"].shape)
                input = batch["input"].to(device).float()
                labels = batch["labels"].to(device)
                labels = batch["logits"].to(device)
                mask = batch["mask"].to(device)

                output = probe(input, mask)

                loss = criterion(output, labels)
                val_loss += loss.item() * len(batch["input"])

                labels = batch["labels"]

                preds = output.argmax(dim=1)  # Assuming classification task
                hits = (preds == labels)

                block_wise_hits += hits.sum(dim=0).cpu().numpy()
                total += len(labels)

                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

            block_wise_hits = block_wise_hits / total

            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)

            # Compute F1 score block-wise
            # block_wise_f1 = np.zeros(n_blocks * 2)
            # for i in range(n_blocks * 2):
            #     block_wise_f1[i] = f1_score(all_labels[:, i], all_preds[:, i], average='macro')

            # avg_f1 = block_wise_f1.mean()
            avg_f1 = f1_score(all_labels, all_preds, average='macro')

            val_loss /= total

            print(
                f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Hits: {block_wise_hits.mean():.4f}, F1: {avg_f1:.4f}, Val Loss: {val_loss:.4f}")

            # Early Stopping Check
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    return block_wise_hits, best_f1

In [36]:
from torch.nn.parallel import DataParallel

n_layer = total_layers - 1

n_prev_tokens = 100
n_shift_tokens= 0

train_dataset = StepProbeDataset(
   train_items, n_layer, n_prev_tokens, n_shift_tokens)
test_dataset = StepProbeDataset(test_items, n_layer, n_prev_tokens, n_shift_tokens)

print(len(train_dataset))

n_dim = 5120
probe = GRUProbe(n_dim, 1000, n_blocks).to(device)
# probe = MLHAProbe(n_dim, n_dim, 40, n_blocks).to(device)

188455


In [ ]:

block_wise_hits, best_f1 = train_probe(
    probe, train_dataset, test_dataset, patience=10, lr=1e-4)

print(best_f1)

Epoch 0, Train Loss: 24.2481, Hits: 0.3398, F1: 0.2023, Val Loss: 6.4483
Epoch 1, Train Loss: 5.2766, Hits: 0.5451, F1: 0.5189, Val Loss: 4.4492
Epoch 2, Train Loss: 3.7743, Hits: 0.5888, F1: 0.5699, Val Loss: 3.6384
Epoch 3, Train Loss: 3.1438, Hits: 0.6087, F1: 0.5925, Val Loss: 3.3168
Epoch 4, Train Loss: 2.7919, Hits: 0.6204, F1: 0.5990, Val Loss: 3.2009
Epoch 5, Train Loss: 2.5330, Hits: 0.6279, F1: 0.6098, Val Loss: 3.0802
Epoch 6, Train Loss: 2.3268, Hits: 0.6302, F1: 0.6150, Val Loss: 3.0008
Epoch 7, Train Loss: 2.1530, Hits: 0.6416, F1: 0.6235, Val Loss: 2.9125
Epoch 8, Train Loss: 2.0062, Hits: 0.6412, F1: 0.6287, Val Loss: 2.8542
Epoch 9, Train Loss: 1.8751, Hits: 0.6544, F1: 0.6397, Val Loss: 2.8036
Epoch 10, Train Loss: 1.7520, Hits: 0.6559, F1: 0.6374, Val Loss: 2.7780


KeyboardInterrupt: 

: 